# Optuna Merged Viewer

`optuna_jh.db`, `optuna_yi.db`, `optuna_yjs.db` 3개의 DB를 한 번에 읽어 study/trial을 병합한다.
어떤 실험(study)들이 있는지 overview를 확인한다.

In [1]:
import os, sys

try:
    import google.colab
    if not os.path.exists("/content/project/setup.py"):
        os.system("pip install -q gdown")
        os.system("gdown --id 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip")
        os.system("unzip -qo /content/code.zip -d /content/project")
        os.makedirs("/content/project/0_data", exist_ok=True)
        os.system("gdown --id 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip")
        os.system("unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data")
        os.remove("/content/project/0_data/dataset.zip")
    sys.path.insert(0, "/content/project")
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

setup 완료


## 1. 3개 DB 로드 & study summary 병합

각 study에 `owner`(jh/yi/yjs) 컬럼을 붙여 어느 DB에서 온 실험인지 구분한다.

In [2]:
import optuna
import pandas as pd
from pathlib import Path

optuna.logging.set_verbosity(optuna.logging.WARNING)

DB_FILES = {
    "jh":  "optuna_jh.db",
    "yi":  "optuna_yi.db",
    "yjs": "optuna_yjs.db",
}

all_summaries = {}   # owner -> list of summaries
rows = []
for owner, fname in DB_FILES.items():
    db_path = Path(fname).resolve()
    if not db_path.exists():
        print(f"[skip] {owner}: {db_path} not found")
        continue
    storage = f"sqlite:///{db_path.as_posix()}"
    summaries = optuna.get_all_study_summaries(storage)
    all_summaries[owner] = (storage, summaries)
    for s in summaries:
        rows.append({
            "owner": owner,
            "study_name": s.study_name,
            "n_trials": s.n_trials,
            "direction": s.direction.name,
            "best_value": s.best_trial.value if s.best_trial else None,
            "datetime_start": s.datetime_start,
        })

study_info = pd.DataFrame(rows).sort_values(["owner", "datetime_start"]).reset_index(drop=True)
print(f"총 DB: {len(all_summaries)} / 총 study: {len(study_info)}")
study_info

총 DB: 3 / 총 study: 9


,owner,study_name,n_trials,direction,best_value,datetime_start
0,jh,3-101-001,440,MINIMIZE,0.005544,2026-04-13 16:54:40.233101
1,jh,3-101-002,54,MINIMIZE,0.005528,2026-04-14 09:50:01.681912
2,yi,1-002-002,15,MINIMIZE,0.005501,2026-04-13 14:14:59.752422
3,yi,1-002-001,5,MINIMIZE,0.005502,2026-04-13 14:54:43.001883
4,yi,1-003-001,10,MINIMIZE,0.001387,2026-04-13 16:45:33.578429
5,yi,1-010-001,480,MINIMIZE,0.001315,2026-04-13 16:54:14.506800
6,yi,1-010-002,67,MINIMIZE,0.001387,2026-04-14 09:48:49.916675
7,yjs,1-100-001,400,MINIMIZE,0.005456,2026-04-13 20:37:21.492065
8,yjs,1-100-002,87,MINIMIZE,0.005994,2026-04-14 09:48:44.335686


### 1-1. 제외할 study 드랍 (하드코딩)

분석에서 제외하고 싶은 study를 `(owner, study_name)` 쌍으로 지정한다.
이후 모든 셀(overview, trial 병합, 조합 매칭)은 드랍된 상태의 `study_info` / `all_summaries`를 사용한다.

In [ ]:
# 제외할 study 목록 (하드코딩)
DROP_STUDIES = [
    ("yi", "1-002-001"),
    ("yi", "1-002-002"),
    ("yi", "1-003-001"),
]

_before = len(study_info)

# 1) study_info에서 드랍
drop_mask = study_info.set_index(["owner", "study_name"]).index.isin(DROP_STUDIES)
study_info = study_info[~drop_mask].reset_index(drop=True)

# 2) all_summaries에서도 동일하게 드랍 (downstream의 load_study / trials_dataframe이 제외된 상태로 돌도록)
drop_set = set(DROP_STUDIES)
for owner in list(all_summaries.keys()):
    storage, summaries = all_summaries[owner]
    filtered = [s for s in summaries if (owner, s.study_name) not in drop_set]
    all_summaries[owner] = (storage, filtered)

print(f"드랍: {_before - len(study_info)}개 / 남은 study: {len(study_info)}")
study_info

## 2. Overview: owner별 실험 수 & best value

누가 어떤 study들을 돌렸는지 한눈에 본다.

In [3]:
# owner별 요약
owner_summary = (
    study_info.groupby("owner")
    .agg(
        n_studies=("study_name", "count"),
        total_trials=("n_trials", "sum"),
        best_value=("best_value", "min"),
    )
    .reset_index()
)
print("[owner별 요약]")
display(owner_summary)

# 전체 study 목록 (best_value 오름차순)
print("\n[study 목록 — best_value 오름차순]")
study_info.sort_values("best_value", na_position="last").reset_index(drop=True)

[owner별 요약]


,owner,n_studies,total_trials,best_value
0,jh,2,494,0.005528
1,yi,5,577,0.001315
2,yjs,2,487,0.005456



[study 목록 — best_value 오름차순]


,owner,study_name,n_trials,direction,best_value,datetime_start
0,yi,1-010-001,480,MINIMIZE,0.001315,2026-04-13 16:54:14.506800
1,yi,1-003-001,10,MINIMIZE,0.001387,2026-04-13 16:45:33.578429
2,yi,1-010-002,67,MINIMIZE,0.001387,2026-04-14 09:48:49.916675
3,yjs,1-100-001,400,MINIMIZE,0.005456,2026-04-13 20:37:21.492065
4,yi,1-002-002,15,MINIMIZE,0.005501,2026-04-13 14:14:59.752422
5,yi,1-002-001,5,MINIMIZE,0.005502,2026-04-13 14:54:43.001883
6,jh,3-101-002,54,MINIMIZE,0.005528,2026-04-14 09:50:01.681912
7,jh,3-101-001,440,MINIMIZE,0.005544,2026-04-13 16:54:40.233101
8,yjs,1-100-002,87,MINIMIZE,0.005994,2026-04-14 09:48:44.335686


## 3. 모든 trial을 하나의 DataFrame으로 병합

study_name이 owner별로 겹칠 수 있으므로 `owner`, `study_key`(= `owner/study_name`) 컬럼을 추가한다.

In [4]:
frames = []
for owner, (storage, summaries) in all_summaries.items():
    for s in summaries:
        study = optuna.load_study(study_name=s.study_name, storage=storage)
        df = study.trials_dataframe()
        if df.empty:
            continue
        df.insert(0, "owner", owner)
        df.insert(1, "study_name", s.study_name)
        df.insert(2, "study_key", f"{owner}/{s.study_name}")
        frames.append(df)

trials_df = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
print(f"전체 trial shape: {trials_df.shape}")
trials_df.head()

전체 trial shape: (1558, 66)


,owner,study_name,study_key,number,value,datetime_start,datetime_complete,duration,params_clf_colsample_bytree,params_clf_filter_threshold,...,user_attrs_clf_val_recall,user_attrs_n_feat_clean,user_attrs_n_feat_selected,user_attrs_outlier_args,user_attrs_saved_at,user_attrs_selected_cols,user_attrs_train_rmse,user_attrs_val_rmse,system_attrs_fixed_params,state
0,jh,3-101-001,jh/3-101-001,0,0.006123,2026-04-13 16:54:40.233101,2026-04-13 16:55:33.532639,0 days 00:00:53.299538,0.738578,0.25,...,0.0,569.0,570.0,"{'method': 'winsorize', 'lower_pct': 0.0, 'upp...",2026-04-13 16:55,"[X4, X6, X7, X10, X12, X14, X17, X18, X19, X21...",0.006123,0.006331,NaN,COMPLETE
1,jh,3-101-001,jh/3-101-001,1,0.006123,2026-04-13 16:55:33.690055,2026-04-13 16:56:24.209119,0 days 00:00:50.519064,0.826914,0.50,...,0.0,667.0,668.0,"{'method': 'winsorize', 'lower_pct': 0.0, 'upp...",2026-04-13 16:56,"[X4, X5, X6, X7, X8, X9, X10, X11, X12, X13, X...",0.006123,0.006331,NaN,COMPLETE
2,jh,3-101-001,jh/3-101-001,2,0.006123,2026-04-13 16:56:24.355518,2026-04-13 16:57:20.541679,0 days 00:00:56.186161,0.853378,0.45,...,0.0,663.0,664.0,"{'method': 'winsorize', 'lower_pct': 0.0, 'upp...",2026-04-13 16:57,"[X4, X5, X6, X7, X8, X9, X10, X11, X12, X13, X...",0.006123,0.006331,NaN,COMPLETE
3,jh,3-101-001,jh/3-101-001,3,0.005873,2026-04-13 16:57:20.686870,2026-04-13 16:58:43.653218,0 days 00:01:22.966348,0.848614,0.10,...,0.0,569.0,570.0,"{'method': 'winsorize', 'lower_pct': 0.0, 'upp...",2026-04-13 16:58,"[X4, X6, X7, X10, X12, X14, X17, X18, X19, X21...",0.005873,0.006085,NaN,COMPLETE
4,jh,3-101-001,jh/3-101-001,4,0.006123,2026-04-13 16:58:43.803573,2026-04-13 17:00:05.002732,0 days 00:01:21.199159,0.880904,0.10,...,0.0,671.0,672.0,"{'method': 'winsorize', 'lower_pct': 0.0, 'upp...",2026-04-13 17:00,"[X1, X4, X5, X6, X7, X8, X9, X10, X11, X12, X1...",0.006123,0.006331,NaN,COMPLETE


## 4. 전체 DB 통합 Top-N

3개 DB의 모든 COMPLETE trial 중 성능이 가장 좋은 trial을 뽑는다.

In [5]:
TOP_N = 20

complete = trials_df[trials_df["state"] == "COMPLETE"].copy()
# direction은 MINIMIZE 가정 (study_info에서 확인 후 다르면 수정)
top_trials = (
    complete.sort_values("value", ascending=True)
    .loc[:, ["owner", "study_name", "number", "value", "datetime_start"]]
    .head(TOP_N)
    .reset_index(drop=True)
)
top_trials

,owner,study_name,number,value,datetime_start
0,yi,1-010-001,435,0.001315,2026-04-14 04:09:46.647701
1,yi,1-010-001,419,0.001319,2026-04-14 03:37:28.889379
2,yi,1-010-001,397,0.001323,2026-04-14 02:56:42.144080
3,yi,1-010-001,383,0.001325,2026-04-14 02:29:17.623201
4,yi,1-010-001,394,0.001326,2026-04-14 02:49:18.458754
5,yi,1-010-001,352,0.001327,2026-04-14 01:26:41.045347
6,yi,1-010-001,412,0.001327,2026-04-14 03:24:36.606459
7,yi,1-010-001,381,0.001328,2026-04-14 02:25:16.581325
8,yi,1-010-001,326,0.001328,2026-04-14 00:35:04.348742
9,yi,1-010-001,321,0.001328,2026-04-14 00:24:32.652328


## 5. 특정 study만 필터링

분석에 쓸 study만 남긴다. `(owner, study_name)` 쌍으로 지정하면 정렬 순서가 바뀌어도 안전하다.

- `jh / 3-101-001`
- `yi / 1-010-001`
- `yjs / 1-100-001`

In [6]:
KEEP_STUDIES = [
    ("jh",  "3-101-001"),
    ("yi",  "1-010-001"),
    ("yjs", "1-100-001"),
]

keep_idx = pd.MultiIndex.from_tuples(KEEP_STUDIES, names=["owner", "study_name"])

study_mask = study_info.set_index(["owner", "study_name"]).index.isin(keep_idx)
study_info_sel = study_info[study_mask].reset_index(drop=True)

trial_mask = trials_df.set_index(["owner", "study_name"]).index.isin(keep_idx)
trials_sel = trials_df[trial_mask].reset_index(drop=True)

print(f"선택된 study: {len(study_info_sel)} / 전체: {len(study_info)}")
print(f"선택된 trial: {len(trials_sel)} / 전체: {len(trials_df)}")
display(study_info_sel)

선택된 study: 3 / 전체: 9
선택된 trial: 1320 / 전체: 1558


,owner,study_name,n_trials,direction,best_value,datetime_start
0,jh,3-101-001,440,MINIMIZE,0.005544,2026-04-13 16:54:40.233101
1,yi,1-010-001,480,MINIMIZE,0.001315,2026-04-13 16:54:14.506800
2,yjs,1-100-001,400,MINIMIZE,0.005456,2026-04-13 20:37:21.492065


## 6. 조합표(experiment_combinations.xlsx) 매칭

`experiment_combinations.xlsx`의 36개 조합 각각이 **실행됐는지 / 성능이 얼마인지**를 한눈에 본다.

**매칭 키** (study `user_attrs` 기준):

| 조합표 컬럼 | user_attrs 위치 |
|---|---|
| `run_clf` | `pipeline_config.run_clf` |
| `reg_level` | `pipeline_config.reg_level` |
| `TARGET_TRANSFORM` | `target_transform` |
| `CLIP_Y_EXTREME` | `clip_y_extreme` |
| `clf_output` | `pipeline_config.clf_output` (run_clf=False 이면 무시) |

- 한 조합에 매칭되는 study가 여러 개면 **val_rmse가 가장 낮은 것**을 채택
- 날짜는 study `datetime_start`에서 추출
- val_rmse/test_rmse/user는 study user_attrs의 `final_val_rmse`/`final_test_rmse`/`user`

In [7]:
# 6-1. 조합표 로드 + 각 study의 config를 user_attrs에서 추출
combos = pd.read_excel("experiment_combinations.xlsx")
print(f"조합 수: {len(combos)} | 컬럼: {combos.columns.tolist()}")

cfg_rows = []
for owner, (storage, summaries) in all_summaries.items():
    for s in summaries:
        study = optuna.load_study(study_name=s.study_name, storage=storage)
        ua = dict(study.user_attrs)
        pc = ua.get("pipeline_config", {}) or {}
        cfg_rows.append({
            "owner": owner,
            "study_name": s.study_name,
            "exp_id": ua.get("exp_id", s.study_name),
            "run_clf": pc.get("run_clf"),
            "reg_level": pc.get("reg_level"),
            "TARGET_TRANSFORM": ua.get("target_transform"),
            "CLIP_Y_EXTREME": ua.get("clip_y_extreme"),
            "clf_output": pc.get("clf_output") if pc.get("run_clf") else None,
            "val_rmse": ua.get("final_val_rmse"),
            "test_rmse": ua.get("final_test_rmse"),
            "user": ua.get("user", owner),
            "날짜": s.datetime_start.date() if s.datetime_start is not None else None,
        })

study_cfg = pd.DataFrame(cfg_rows)
print(f"study config rows: {len(study_cfg)}")
study_cfg

조합 수: 36 | 컬럼: ['run_clf', 'reg_level', 'TARGET_TRANSFORM', 'CLIP_Y_EXTREME', 'clf_output', '날짜', 'val_rmse', 'test_rmse', 'user']
study config rows: 9


,owner,study_name,exp_id,run_clf,reg_level,TARGET_TRANSFORM,CLIP_Y_EXTREME,clf_output,val_rmse,test_rmse,user,날짜
0,jh,3-101-001,3-101-001,True,position,none,True,proba,0.005757,0.008447,jh,2026-04-13
1,jh,3-101-002,3-101-002,False,position,none,True,None,NaN,NaN,jh,2026-04-14
2,yi,1-002-002,1-002-002,True,position,log1p,True,proba,0.005786,0.008467,yi,2026-04-13
3,yi,1-002-001,1-002-001,True,position,log1p,True,proba,0.005785,0.008467,yi,2026-04-13
4,yi,1-003-001,1-003-001,True,position,yeo-johnson,True,binary,NaN,NaN,yi,2026-04-13
5,yi,1-010-001,1-010-001,True,position,yeo-johnson,True,binary,0.006264,0.008833,yi,2026-04-13
6,yi,1-010-002,1-010-002,False,position,yeo-johnson,True,None,NaN,NaN,yi,2026-04-14
7,yjs,1-100-001,1-100-001,True,position,log1p,True,proba,0.005744,0.008427,yjs,2026-04-13
8,yjs,1-100-002,1-100-002,True,position,log1p,True,binary,NaN,NaN,yjs,2026-04-14


In [8]:
# 6-2. 조합표와 study config를 매칭 → 36행 결과표
KEY_COLS = ["run_clf", "reg_level", "TARGET_TRANSFORM", "CLIP_Y_EXTREME", "clf_output"]

# clf_output의 NaN(= run_clf=False 일 때)은 merge가 안되므로 sentinel로 치환
NA_SENTINEL = "__NA__"
combos_m = combos[KEY_COLS].copy()
combos_m["clf_output"] = combos_m["clf_output"].fillna(NA_SENTINEL)

study_cfg_m = study_cfg.copy()
study_cfg_m["clf_output"] = study_cfg_m["clf_output"].fillna(NA_SENTINEL)

# 조합 단위로 best(val_rmse 최소)만 남김 — val_rmse가 있는 study만 대상
best_per_combo = (
    study_cfg_m.dropna(subset=["val_rmse"])
    .sort_values("val_rmse", ascending=True)
    .drop_duplicates(subset=KEY_COLS, keep="first")
)

# 한 조합에 붙은 study 수 (val_rmse 유무 무관)
match_count = (
    study_cfg_m.groupby(KEY_COLS, dropna=False)
    .size()
    .rename("n_studies")
    .reset_index()
)

result = (
    combos_m.merge(
        best_per_combo[KEY_COLS + ["exp_id", "날짜", "val_rmse", "test_rmse", "user"]],
        on=KEY_COLS,
        how="left",
    )
    .merge(match_count, on=KEY_COLS, how="left")
)
result["n_studies"] = result["n_studies"].fillna(0).astype(int)
result["clf_output"] = result["clf_output"].replace(NA_SENTINEL, None)

# done 상태 3단계
#   O : 실험 완료 (final_val_rmse 존재)
#   △ : study는 있으나 final_val_rmse 미기록 (진행중/rerun 미완료)
#   X : 시도 없음
def _status(row):
    if pd.notna(row["val_rmse"]):
        return "O"
    if row["n_studies"] > 0:
        return "△"
    return "X"

result["done"] = result.apply(_status, axis=1)

display_cols = KEY_COLS + ["done", "n_studies", "exp_id", "날짜", "val_rmse", "test_rmse", "user"]

n_done = (result["done"] == "O").sum()
n_partial = (result["done"] == "△").sum()
n_todo = (result["done"] == "X").sum()
print(f"완료(O): {n_done} | 진행중(△): {n_partial} | 미시도(X): {n_todo} | 총 {len(result)} 조합")

# O → △ → X 순, 동일 상태 내에서는 val_rmse 오름차순
status_order = {"O": 0, "△": 1, "X": 2}
result["_ord"] = result["done"].map(status_order)
result_sorted = (
    result.sort_values(["_ord", "val_rmse"], ascending=[True, True])
    .drop(columns="_ord")
    .reset_index(drop=True)
)
result_sorted[display_cols]

완료(O): 3 | 진행중(△): 3 | 미시도(X): 30 | 총 36 조합


,run_clf,reg_level,TARGET_TRANSFORM,CLIP_Y_EXTREME,clf_output,done,n_studies,exp_id,날짜,val_rmse,test_rmse,user
0,True,position,log1p,True,proba,O,3,1-100-001,2026-04-13,0.005744,0.008427,yjs
1,True,position,none,True,proba,O,1,3-101-001,2026-04-13,0.005757,0.008447,jh
2,True,position,yeo-johnson,True,binary,O,2,1-010-001,2026-04-13,0.006264,0.008833,yi
3,True,position,log1p,True,binary,△,1,NaN,NaN,NaN,NaN,NaN
4,False,position,yeo-johnson,True,None,△,1,NaN,NaN,NaN,NaN,NaN
5,False,position,none,True,None,△,1,NaN,NaN,NaN,NaN,NaN
6,True,unit,log1p,True,proba,X,0,NaN,NaN,NaN,NaN,NaN
7,True,unit,log1p,True,binary,X,0,NaN,NaN,NaN,NaN,NaN
8,True,unit,log1p,False,proba,X,0,NaN,NaN,NaN,NaN,NaN
9,True,unit,log1p,False,binary,X,0,NaN,NaN,NaN,NaN,NaN
